In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('Raw/diabetic_data.csv', na_values='?')

In [3]:
cols_to_drop= ['weight', 'max_glu_serum', 'A1Cresult','medical_specialty', 'payer_code']
df=df.drop(columns=cols_to_drop)

In [4]:
cols_to_fill=['race', 'diag_1', 'diag_2', 'diag_3']
df[cols_to_fill]=df[cols_to_fill].fillna('Unknown')

In [5]:
df['readmitted_binary']=(df['readmitted'] == '<30').astype(int)
print(f"Data Loaded: {df.shape}")

Data Loaded: (101766, 46)


In [6]:
## Age Encoding

In [7]:
age_map= {'[0-10)': 5,'[10-20)': 15, '[20-30)' : 25, '[30-40)' : 35, '[40-50)' : 45, '[50-60)' : 55,
    '[60-70)' : 65, '[70-80)' : 75, '[80-90)' : 85, '[90-100)': 95 }
df['age_numeric']= df['age'].map(age_map)
print(df[['age', 'age_numeric']].drop_duplicates().sort_values('age_numeric'))
print(f"\nMissing values in age_numeric: {df['age_numeric'].isnull().sum()}")

        age  age_numeric
0    [0-10)            5
1   [10-20)           15
2   [20-30)           25
3   [30-40)           35
4   [40-50)           45
5   [50-60)           55
6   [60-70)           65
7   [70-80)           75
8   [80-90)           85
9  [90-100)           95

Missing values in age_numeric: 0


In [8]:
## Diagnosis Grouping (Patients having multiple conditions simultaneously is Comorbidity)

In [9]:
def group_diagnosis(code):
    try:
        c = str(code)
        if c.startswith('V') or c.startswith('E'):
            return 'Other'
        num = float(c)
        if 390 <= num <= 459 or num == 785:
            return 'Circulatory'
        elif 460 <= num <= 519 or num == 786:
            return 'Respiratory'
        elif 520 <= num <= 579 or num == 787:
            return 'Digestive'
        elif c.startswith('250'):
            return 'Diabetes'
        elif 800 <= num <= 999:
            return 'Injury'
        elif 710 <= num <= 739:
            return 'Musculoskeletal'
        elif 580 <= num <= 629 or num == 788:
            return 'Genitourinary'
        elif 140 <= num <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except:
        return 'Other'

df['diag_1_group'] = df['diag_1'].apply(group_diagnosis)
df['diag_2_group'] = df['diag_2'].apply(group_diagnosis)
df['diag_3_group'] = df['diag_3'].apply(group_diagnosis)

print("Diagnosis 1 groups:")
print(df['diag_1_group'].value_counts())
print("\nDiagnosis 2 groups:")
print(df['diag_2_group'].value_counts())
print("\nDiagnosis 3 groups:")
print(df['diag_3_group'].value_counts())

Diagnosis 1 groups:
diag_1_group
Circulatory        30437
Other              18193
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Name: count, dtype: int64

Diagnosis 2 groups:
diag_2_group
Circulatory        31881
Other              26911
Diabetes           12794
Respiratory        10895
Genitourinary       8376
Digestive           4170
Neoplasms           2547
Injury              2428
Musculoskeletal     1764
Name: count, dtype: int64

Diagnosis 3 groups:
diag_3_group
Other              30618
Circulatory        30306
Diabetes           17157
Respiratory         7358
Genitourinary       6680
Digestive           3930
Injury              1946
Musculoskeletal     1915
Neoplasms           1856
Name: count, dtype: int64


In [10]:
## Comorbidity Score
###Comorbidity scoring is a standard technique in health analytics and is used by Insurance companies for risk scoring, Hospitals for discharge planning, Government for Medicare/Medicaid reimbursemen

In [11]:
def count_comorbidities(row):
    diagnoses=[row['diag_1_group'], row['diag_2_group'], row['diag_3_group']]
    diagnoses= [d for d in diagnoses if d not in ['Other','Unknown']]
    return len(set(diagnoses))

df['comorbidity_score']= df.apply(count_comorbidities, axis=1)
print("Comorbidity Score Distribution:")
print(df['comorbidity_score'].value_counts().sort_index())
print(f"\nAverage comorbidity score: {df['comorbidity_score'].mean():.2f}")

Comorbidity Score Distribution:
comorbidity_score
0     2235
1    33395
2    51895
3    14241
Name: count, dtype: int64

Average comorbidity score: 1.77


In [12]:
## Comorbidity score Vs Readmission Rate

In [13]:
comorbidity_readmit = df.groupby('comorbidity_score')['readmitted_binary'].agg(['mean','count']).reset_index()
comorbidity_readmit.columns = ['Comorbidity Score', 'Readmission Rate', 'Count']
comorbidity_readmit['Readmission Rate'] = comorbidity_readmit['Readmission Rate'] * 100
print("\nReadmission Rate by Comorbidity Score:")
print(comorbidity_readmit)


Readmission Rate by Comorbidity Score:
   Comorbidity Score  Readmission Rate  Count
0                  0          9.932886   2235
1                  1         10.977691  33395
2                  2         11.390307  51895
3                  3         10.940243  14241


In [14]:
##Prior Visits Score(In, out, Emergency)

In [15]:
###Combining all 3 visit types into one weighted score based on their predictive strength found in EDA:
####- Inpatient  × 3 (strongest predictor)
####- Emergency  × 2 (strong predictor)  
####- Outpatient × 1 (weak predictor)

In [16]:
df['prior_visit_score'] = ((df['number_inpatient']  * 3) +
                            (df['number_emergency']  * 2) +
                            (df['number_outpatient'] * 1))
print("Prior Visit Score Distribution:")
print(df['prior_visit_score'].describe())

Prior Visit Score Distribution:
count    101766.000000
mean          2.671727
std           4.964195
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max         160.000000
Name: prior_visit_score, dtype: float64


In [17]:
### Find who has score 160+

In [18]:
high_scores = df[df['prior_visit_score'] >= 50][['number_inpatient', 
                                                   'number_emergency',
                                                   'number_outpatient',
                                                   'prior_visit_score']]
print(high_scores.sort_values('prior_visit_score', ascending=False).head(10))

       number_inpatient  number_emergency  number_outpatient  \
83844                 2                76                  2   
88897                 3                63                  2   
88079                 2                64                  2   
94341                 4                54                  3   
32661                 4                42                  3   
77142                 2                46                  0   
87121                14                22                  0   
85570                14                22                  0   
84248                13                22                  0   
61923                19                13                  0   

       prior_visit_score  
83844                160  
88897                137  
88079                136  
94341                123  
32661                 99  
77142                 98  
87121                 86  
85570                 86  
84248                 83  
61923                 83 

In [19]:
### 99th percentile ( Investigation revealed that 99 percent of them has 3 visits and the others were extreme values confirming outliers.

In [20]:
print(df['number_emergency'].quantile(0.99))

3.0


In [21]:
## Capped Prior Visit Score

In [22]:
emergency_cap = df['number_emergency'].quantile(0.99)
print(f"Capping number_emergency at: {emergency_cap}")
df['number_emergency'] = df['number_emergency'].clip(upper=emergency_cap)

# Recalculate prior visit score
df['prior_visit_score'] = ((df['number_inpatient']  * 3) +
                            (df['number_emergency']  * 2) +
                            (df['number_outpatient'] * 1))

print(f"New max score: {df['prior_visit_score'].max()}")
print(df['prior_visit_score'].describe())

Capping number_emergency at: 3.0
New max score: 65
count    101766.000000
mean          2.605300
std           4.559744
min           0.000000
25%           0.000000
50%           0.000000
75%           3.000000
max          65.000000
Name: prior_visit_score, dtype: float64


In [23]:
## Medication Change Flags(Binary Encoding: True- Up/Down, False- Steady/No)

In [24]:
# Insulin flag
df['insulin_changed'] = df['insulin'].isin(['Up', 'Down']).astype(int)

# Other medication columns
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone',
            'metformin-rosiglitazone', 'metformin-pioglitazone']

# Check which columns actually exist in our dataframe
med_cols = [col for col in med_cols if col in df.columns]

# Any medication changed flag
df['any_med_changed'] = df[med_cols].isin(['Up', 'Down']).any(axis=1).astype(int)
print(df['insulin_changed'].value_counts())
print(df['any_med_changed'].value_counts())

for col in ['insulin_changed', 'any_med_changed']:
    rate = (df.groupby(col)['readmitted_binary'].mean() * 100).round(2)
    print(rate)

insulin_changed
0    78232
1    23534
Name: count, dtype: int64
any_med_changed
0    74063
1    27703
Name: count, dtype: int64
insulin_changed
0    10.47
1    13.46
Name: readmitted_binary, dtype: float64
any_med_changed
0    10.45
1    13.06
Name: readmitted_binary, dtype: float64


In [25]:
###Insulin changes show slightly stronger signal than general medication changes, Both are valuable predictors.

In [26]:
## Encode Categorical Variables

In [27]:
### Check and Drop columns

In [28]:
print(df.select_dtypes(include='object').columns.tolist())

['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'diag_1_group', 'diag_2_group', 'diag_3_group']


In [29]:
cols_to_drop = ['encounter_id', 'patient_nbr','age', 'diag_1', 'diag_2', 'diag_3', 'readmitted']
cols_to_drop = [d for d in cols_to_drop if d in df.columns] 
df = df.drop(columns=cols_to_drop)

In [30]:
med_cols_to_drop = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',  'glipizide', 'glyburide', 'tolbutamide', 
                    'pioglitazone', 'rosiglitazone', 'acarbose','miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 
                    'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',  'metformin-pioglitazone']
med_cols_to_drop = [c for c in med_cols_to_drop if c in df.columns]
df = df.drop(columns=med_cols_to_drop)

In [31]:
df.shape

(101766, 24)

In [32]:
### Binary Encoding 

In [33]:
binary_map = {
    'gender'    : {'Male': 0, 'Female': 1},
    'change'    : {'No': 0, 'Ch': 1},
    'diabetesMed': {'No': 0, 'Yes': 1}}

for col, mapping in binary_map.items():
    df[col] = df[col].map(mapping)

In [34]:
## One Hot Encoding

In [35]:
nominal_cols = ['race', 'diag_1_group', 'diag_2_group', 'diag_3_group']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

In [36]:
## Convert all boolean columns to integer

In [37]:
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

In [38]:
print(f"Missing values before: {df.isnull().sum().sum()}")

Missing values before: 3


In [39]:
df = df.dropna()

In [40]:
df.shape

(101763, 53)

In [41]:
print(df.dtypes.value_counts())
print("\nNon-numeric columns:")
print(df.select_dtypes(exclude=['int64', 'float64']).columns.tolist())

int64      52
float64     1
Name: count, dtype: int64

Non-numeric columns:
[]


In [ ]:
## Save as processed csv

In [48]:
df.to_csv('Raw/diabetic_processed.csv', index=False)

print(f"Final shape: {df.shape}")
print(f"Total features: {df.shape[1] -1}")
print(f"Target column: readmitted_binary")
print(f"\nFeature summary:")
print(df.describe().T[['mean', 'min', 'max']].round(2))

Final shape: (101763, 53)
Total features: 52
Target column: readmitted_binary

Feature summary:
                               mean  min    max
gender                         0.54  0.0    1.0
admission_type_id              2.02  1.0    8.0
discharge_disposition_id       3.72  1.0   28.0
admission_source_id            5.75  1.0   25.0
time_in_hospital               4.40  1.0   14.0
num_lab_procedures            43.10  1.0  132.0
num_procedures                 1.34  0.0    6.0
num_medications               16.02  1.0   81.0
number_outpatient              0.37  0.0   42.0
number_emergency               0.16  0.0    3.0
number_inpatient               0.64  0.0   21.0
number_diagnoses               7.42  1.0   16.0
change                         0.46  0.0    1.0
diabetesMed                    0.77  0.0    1.0
readmitted_binary              0.11  0.0    1.0
age_numeric                   65.97  5.0   95.0
comorbidity_score              1.77  0.0    3.0
prior_visit_score              2.61  0.0